In [ ]:
import pandas as pd
import numpy as np

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/test.csv')

In [ ]:
train.info()

In [ ]:
train.isna().sum()

In [ ]:
X = train.drop(columns=["id","satisfaction"])
y = train["satisfaction"]

In [19]:
Xt = test.drop(columns=["id"])

In [ ]:
X["Gender"] = X["Gender"].astype("category")
X["Customer Type"] = X["Customer Type"].astype("category")
X["Type of Travel"] = X["Type of Travel"].astype("category")
X["Class"] = X["Class"].astype("category")

(299844, 21)

In [ ]:
X.dtypes

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_val,y_train,y_val = train_test_split(
    X,y,
    test_size = 0.2,
    stratify = y,
    random_state = 42
    

)


In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
from lightgbm import LGBMClassifier 
model = LGBMClassifier(
    n_estimators = 500,
    learning_rate = 0.05,
    num_leaves = 30,
    random_state=42,
    verbosity = 1,
)
model.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import roc_auc_score
y_probs = model.predict_proba(X_val)[:,1]
auc_score = roc_auc_score(y_val,y_probs)


In [ ]:
print(y_probs.shape)
print(auc_score)

In [ ]:
from lightgbm import LGBMClassifier 
from sklearn.model_selection import StratifiedKFold
oof_preds = np.zeros(len(X))
skf = StratifiedKFold(
    n_splits = 3,
    shuffle =True,
    random_state = 42
)

for fold, (train_idx , val_idx) in enumerate(skf.split(X,y),1):
    X_train_fold = X.iloc[train_idx]
    X_val_fold =  X.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]
    model = LGBMClassifier(
         n_estimators = 500,
        learning_rate = 0.05,
        num_leaves = 30,
        random_state=42,
        verbosity = 1,
    )
    model.fit(X_train_fold,y_train_fold)
    fold_probs = model.predict_proba(X_val_fold)[:,1]
    oof_preds[val_idx] = fold_probs
    auc_score = roc_auc_score(y_val_fold,fold_probs)
    print("fold",fold,"score:",auc_score)
    
overall_auc = roc_auc_score(y, oof_preds)
print("Overall_auc",overall_auc)

    

    

In [ ]:
X2 = X.copy()
X2["Travel_class"] = (
    X2["Type of Travel"].astype(str)
    + "_"
    + X2["Class"].astype(str)
)
X2["Travel_class"] = X2["Travel_class"].astype("category")
print(X2.shape)
print(X2["Travel_class"].head())


In [22]:
Xt["Gender"] = pd.Categorical(
    Xt["Gender"],
    categories = X2["Gender"].cat.categories
)
Xt["Customer Type"] = pd.Categorical(
    Xt["Customer Type"],
    categories = X2["Customer Type"].cat.categories
)
Xt["Type of Travel"] = pd.Categorical(
    Xt["Type of Travel"],
    categories = X2["Type of Travel"].cat.categories
)
Xt["Class"] = pd.Categorical(
    Xt["Class"],
    categories = X2["Class"].cat.categories
)

In [23]:
Xt2 = Xt.copy()
Xt2["Travel_class"] = (
    Xt2["Type of Travel"].astype(str)
    + "_"
    + Xt2["Class"].astype(str)
)
Xt2["Travel_class"] = Xt2["Travel_class"].astype("category")

In [30]:
from lightgbm import LGBMClassifier 
from sklearn.model_selection import StratifiedKFold
from lightgbm import early_stopping
oof_preds = np.zeros(len(X2))
test_preds = np.zeros(len(Xt2))
skf = StratifiedKFold(
    n_splits = 3,
    shuffle =True,
    random_state = 42
)

for fold, (train_idx , val_idx) in enumerate(skf.split(X2,y),1):
    X_train_fold = X2.iloc[train_idx]
    X_val_fold =  X2.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]
    model = LGBMClassifier(
         n_estimators = 2500,
        learning_rate = 0.05,
        num_leaves = 65,
        random_state=42,
        verbosity = 1,
    )
    model.fit(X_train_fold,y_train_fold,
             eval_set = [(X_val_fold,y_val_fold)],
             eval_metric = "auc",
             callbacks=[early_stopping(100, verbose=False)])
    print("Fold", fold, "best iteration:", model.best_iteration_)
    fold_probs = model.predict_proba(X_val_fold)[:,1]
    fold_test_probs = model.predict_proba(Xt2)[:, 1]
    test_preds += fold_test_probs / 3
    oof_preds[val_idx] = fold_probs
    auc_score = roc_auc_score(y_val_fold,fold_probs)
    print("fold",fold,"score:",auc_score)
    
overall_auc = roc_auc_score(y, oof_preds)
print("Overall_auc",overall_auc)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 206893, number of negative: 259530
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030796 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 611
[LightGBM] [Info] Number of data points in the train set: 466423, number of used features: 22
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443574 -> initscore=-0.226671
[LightGBM] [Info] Start training from score -0.226671
Fold 1 best iteration: 426
fold 1 score: 0.9578041633180591
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 206892, number of negative: 259531
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overh

In [31]:
print(test_preds.shape)
print(test_preds[:5])
print(test_preds.min(), test_preds.max())

(299844,)
[0.27266497 0.79899059 0.92625558 0.0723457  0.04781279]
0.004012309622420571 0.986575866969106


In [41]:
data = {
    "id": test["id"],
    "Satisfaction": test_preds
}
Submission = pd.DataFrame(data)
Submission.to_csv("Submission.csv",index = False)
print(Submission.shape)
print(Submission.head())


(299844, 2)
       id  Satisfaction
0  699635      0.272665
1  699636      0.798991
2  699637      0.926256
3  699638      0.072346
4  699639      0.047813


In [42]:
#UNDER HERE ARE ALL TEST WITH COMBINNG DIFFERENT FEAUTUREES AND LABLES, ALL FAILED
#I tried with by lableing flight distance to short, medium and long, the results we worse than just using Type travel + class
#I also tried adding Distance group with flight distance which also resulted in much worse results


In [ ]:

custom_stats = X['Flight Distance'].agg([
    'min', 
    'max', 
    'median', 
    lambda x: x.quantile(0.33), 
    lambda x: x.quantile(0.67)
])


custom_stats.index = ['Min', 'Max', 'Median', '33rd Percentile', '67th Percentile']
print(custom_stats)

In [ ]:
X2["Distance_group"]=pd.cut(
        X2["Flight Distance"],
        bins=[0,725,1547,5000],
        labels = ["Short","Medium","Long"]
)
X2["Distance_group"].value_counts()

In [ ]:
X3 = X2.copy()
X3["Distance_TravelType"] =(
    X3["Distance_group"].astype(str)
    + "_"
    + X3["Type of Travel"].astype(str)
)
X3["Distance_TravelType"] = X3["Distance_TravelType"].astype("category")
print(X3["Distance_TravelType"].head())
print(X3["Distance_TravelType"].nunique())
print(X3.shape)

In [ ]:
X4 = X3.drop(columns=["Distance_TravelType"])

In [ ]:
from lightgbm import LGBMClassifier 
from sklearn.model_selection import StratifiedKFold
oof_preds = np.zeros(len(X4))
skf = StratifiedKFold(
    n_splits = 3,
    shuffle =True,
    random_state = 42
)

for fold, (train_idx , val_idx) in enumerate(skf.split(X4,y),1):
    X_train_fold = X4.iloc[train_idx]
    X_val_fold =  X4.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]
    model = LGBMClassifier(
         n_estimators = 500,
        learning_rate = 0.05,
        num_leaves = 30,
        random_state=42,
        verbosity = 1,
    )
    model.fit(X_train_fold,y_train_fold)
    fold_probs = model.predict_proba(X_val_fold)[:,1]
    oof_preds[val_idx] = fold_probs
    auc_score = roc_auc_score(y_val_fold,fold_probs)
    print("fold",fold,"score:",auc_score)
    
overall_auc = roc_auc_score(y, oof_preds)
print("Overall_auc",overall_auc)
